# Ghigliottina AI - Deep Hyperparameter Optimization
Notebook ottimizzato per massimizzare le performance tramite Deep Grid Search.


In [ ]:
import sys
import os
import json
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

if os.path.basename(os.getcwd()) == 'test':
    os.chdir('..')
sys.path.append(os.path.abspath(os.getcwd()))

from scripts.main import SymbolicKnowledgeGraph, stem

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

results_matrix = []


## 1. Caricamento Dataset


In [ ]:
print("[*] Caricamento del dataset...")
try:
    with open('dataset_ghigliottina/train.json', 'r', encoding='utf-8') as f:
        full_train_data = [json.loads(line) for line in f]
    with open('dataset_ghigliottina/test.json', 'r', encoding='utf-8') as f:
        test_data = [json.loads(line) for line in f]
except Exception as e:
    print(f"[ERROR] {e}")

split_idx = int(len(full_train_data) * 0.8)
train_graph_data = full_train_data[:split_idx]
train_ml_data = full_train_data[split_idx:]

print(f"Train Graph: {len(train_graph_data)} samples")
print(f"Train ML (Rerankers): {len(train_ml_data)} samples")
print(f"Test Set: {len(test_data)} samples")


## 2. Inizializzazione LKG (Topologia + NPMI)


In [ ]:
kg = SymbolicKnowledgeGraph()
cache_graph = "grafi/kg_symbolic_xgb.pkl"
if os.path.exists(cache_graph):
    print(f"[*] Loading LKG from {cache_graph}...")
    kg.load_state(cache_graph)
else:
    print("[*] Building LKG...")
    kg.ingest_knowledge('polirematiche_proverbi/demauro.poli', 'DEMAURO_POLI')
    kg.ingest_knowledge('polirematiche_proverbi/polirematiche', 'POLIREMATICHE')
    kg.ingest_knowledge('polirematiche_proverbi/proverbi', 'PROVERBI')
    for item in train_graph_data:
        try:
            gold = item['sol'].lower().strip()
            hints = [item[f'hint{j}'].lower().strip() for j in range(1, 6)]
            for hint in hints:
                kg.graph[stem(hint)].append(('STORICO_GIOCO', f"{hint} {gold}"))
                kg.graph[stem(gold)].append(('STORICO_GIOCO', f"{hint} {gold}"))
        except: continue
    kg.save_state(cache_graph)

print(f"[*] NPMI Tensors loaded: {kg.has_npmi}")


## 3. Pre-computazione Matrici Dati


In [ ]:
def extract_dataset(data, is_test=False):
    X, y = [], []
    words = []
    groups = []
    
    for game_idx, item in enumerate(tqdm(data, desc="Extracting Features")):
        gold = item['sol'].lower().strip()
        hints = [item[f'hint{j}'] for j in range(1, 6)]
        hints_clean = [h.lower().strip() for h in hints]
        hints_stems = [stem(h) for h in hints_clean]
        
        candidates = kg.get_raw_candidates(hints)
        if not candidates: continue
        
        gold_in_cands = False
        for cand in candidates:
            X.append(cand["features"])
            if cand["word"] == gold:
                y.append(1)
                gold_in_cands = True
            else:
                y.append(0)
            
            if is_test:
                words.append(cand["word"])
                groups.append(game_idx)
        
        if not is_test and not gold_in_cands:
            dummy_gold_dict = {
                "word": gold, "unique_hints": 0, "evidences_count": 0,
                "symbolic_score": 0.0, "count_proverbi": 0, "count_poli": 0,
                "count_storico": 0, "node_degree": 0
            }
            dummy_gold_feat = kg._extract_features(dummy_gold_dict, hints_clean, hints_stems)
            X.append(dummy_gold_feat)
            y.append(1)
            
    if is_test:
        return np.array(X, dtype=np.float32), np.array(y, dtype=int), groups, words
    return np.array(X, dtype=np.float32), np.array(y, dtype=int)

print("Estraggo Training Set ML...")
X_train, y_train = extract_dataset(train_ml_data, is_test=False)
print(f"X_train shape: {X_train.shape}")

print("\nEstraggo Test Set...")
X_test, y_test, test_groups, test_words = extract_dataset(test_data, is_test=True)
print(f"X_test shape: {X_test.shape}")


## 4. Helper Function: Valutazione Raggruppata


In [ ]:
def evaluate_model(model, X_test, test_groups, test_words, test_data):
    probs = model.predict_proba(X_test)[:, 1]
    
    games = {}
    for i in range(len(probs)):
        g_idx = test_groups[i]
        if g_idx not in games:
            games[g_idx] = []
        games[g_idx].append((test_words[i], probs[i]))
        
    top1, top5, mrr_sum = 0, 0, 0.0
    total = len(test_data)
    
    for g_idx, cands in games.items():
        gold = test_data[g_idx]['sol'].lower().strip()
        cands.sort(key=lambda x: x[1], reverse=True)
        pred_words = [c[0] for c in cands[:5]]
        
        if pred_words:
            if pred_words[0] == gold: top1 += 1
            if gold in pred_words:
                top5 += 1
                mrr_sum += 1.0 / (pred_words.index(gold) + 1)
                
    return (top1/total)*100, (top5/total)*100, (mrr_sum/total)


## 5. Deep Hyperparameter Tuning


### XGBoost Tuning


In [ ]:
print("Deep Tuning XGBoost...")
param_dist_xgb = {
    'max_depth': [3, 4, 5, 6, 7, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'n_estimators': [100, 200, 300, 500],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}
xgb_clf = xgb.XGBClassifier(n_jobs=-1, eval_metric='logloss')
rs_xgb = RandomizedSearchCV(xgb_clf, param_dist_xgb, n_iter=30, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

start_time = time.time()
rs_xgb.fit(X_train, y_train)
elapsed = time.time() - start_time

best_xgb = rs_xgb.best_estimator_
print(f"Best XGB Params: {rs_xgb.best_params_}")
acc1, acc5, mrr = evaluate_model(best_xgb, X_test, test_groups, test_words, test_data)
print(f"XGBoost -> Acc@1: {acc1:.2f}% | Acc@5: {acc5:.2f}% | MRR: {mrr:.4f} | Tuning Time: {elapsed:.2f}s")

results_matrix.append({
    "Reranker": "XGBoost", "Acc@1": acc1, "Acc@5": acc5, "MRR": mrr, 
    "Best Params": str(rs_xgb.best_params_), "Tuning Time (s)": elapsed
})


### LightGBM Tuning


In [ ]:
print("Deep Tuning LightGBM...")
param_dist_lgb = {
    'max_depth': [-1, 4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [100, 200, 300, 500],
    'num_leaves': [15, 31, 63, 127],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}
lgb_clf = lgb.LGBMClassifier(n_jobs=-1, verbose=-1)
rs_lgb = RandomizedSearchCV(lgb_clf, param_dist_lgb, n_iter=30, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

start_time = time.time()
rs_lgb.fit(X_train, y_train)
elapsed = time.time() - start_time

best_lgb = rs_lgb.best_estimator_
print(f"Best LGB Params: {rs_lgb.best_params_}")
acc1, acc5, mrr = evaluate_model(best_lgb, X_test, test_groups, test_words, test_data)
print(f"LightGBM -> Acc@1: {acc1:.2f}% | Acc@5: {acc5:.2f}% | MRR: {mrr:.4f} | Tuning Time: {elapsed:.2f}s")

results_matrix.append({
    "Reranker": "LightGBM", "Acc@1": acc1, "Acc@5": acc5, "MRR": mrr, 
    "Best Params": str(rs_lgb.best_params_), "Tuning Time (s)": elapsed
})


### CatBoost Tuning


In [ ]:
print("Deep Tuning CatBoost...")
param_dist_cat = {
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'iterations': [100, 300, 500],
    'l2_leaf_reg': [1, 3, 5, 7, 9]
}
cat_clf = CatBoostClassifier(verbose=False, thread_count=-1)
rs_cat = RandomizedSearchCV(cat_clf, param_dist_cat, n_iter=30, cv=3, scoring='accuracy', n_jobs=1, random_state=42)

start_time = time.time()
rs_cat.fit(X_train, y_train)
elapsed = time.time() - start_time

best_cat = rs_cat.best_estimator_
print(f"Best CatBoost Params: {rs_cat.best_params_}")
acc1, acc5, mrr = evaluate_model(best_cat, X_test, test_groups, test_words, test_data)
print(f"CatBoost -> Acc@1: {acc1:.2f}% | Acc@5: {acc5:.2f}% | MRR: {mrr:.4f} | Tuning Time: {elapsed:.2f}s")

results_matrix.append({
    "Reranker": "CatBoost", "Acc@1": acc1, "Acc@5": acc5, "MRR": mrr, 
    "Best Params": str(rs_cat.best_params_), "Tuning Time (s)": elapsed
})


### Random Forest Tuning


In [ ]:
print("Deep Tuning RandomForest...")
param_dist_rf = {
    'max_depth': [6, 10, 15, 20, None],
    'n_estimators': [100, 200, 300, 500],
    'min_samples_split': [2, 5, 10]
}
rf_clf = RandomForestClassifier(n_jobs=-1, random_state=42)
rs_rf = RandomizedSearchCV(rf_clf, param_dist_rf, n_iter=30, cv=3, scoring='accuracy', n_jobs=-1, random_state=42)

start_time = time.time()
rs_rf.fit(X_train, y_train)
elapsed = time.time() - start_time

best_rf = rs_rf.best_estimator_
print(f"Best RF Params: {rs_rf.best_params_}")
acc1, acc5, mrr = evaluate_model(best_rf, X_test, test_groups, test_words, test_data)
print(f"RandomForest -> Acc@1: {acc1:.2f}% | Acc@5: {acc5:.2f}% | MRR: {mrr:.4f} | Tuning Time: {elapsed:.2f}s")

results_matrix.append({
    "Reranker": "RandomForest", "Acc@1": acc1, "Acc@5": acc5, "MRR": mrr, 
    "Best Params": str(rs_rf.best_params_), "Tuning Time (s)": elapsed
})


## 6. Risultati Finali e Matrice Comparativa DEEP HPO


In [ ]:
df = pd.DataFrame(results_matrix)
df.sort_values(by="Acc@1", ascending=False, inplace=True)

df_print = df.copy()
df_print['Acc@1'] = df_print['Acc@1'].apply(lambda x: f"{x:.2f}%")
df_print['Acc@5'] = df_print['Acc@5'].apply(lambda x: f"{x:.2f}%")
df_print['MRR'] = df_print['MRR'].apply(lambda x: f"{x:.4f}")
df_print['Tuning Time (s)'] = df_print['Tuning Time (s)'].apply(lambda x: f"{x:.2f}")

print('\n\n' + '='*80)
print(' FINAL DEEP HPO EVALUATION MATRIX ')
print('='*80)
print(df_print.to_markdown(index=False))

df.to_csv("deep_hpo_evaluation_results.csv", index=False)
df_print


## 7. Analisi Visiva del Vincitore (XGBoost)
Visualizziamo la **Feature Importance** e la **Confusion Matrix** del modello migliore per capire come ha bilanciato le decisioni.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Assicuriamoci che best_xgb sia disponibile
if 'best_xgb' in locals():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1. Feature Importance
    # Se le features hanno nomi numerici, possiamo provare a mapparli logicamente se li conosciamo,
    # ma di default XGBoost li chiama f0, f1, ecc.
    feature_names = ['Unique Hints', 'Evidences Count', 'Symbolic Score', 
                     'Count Proverbi', 'Count Poli', 'Count Storico', 'Node Degree', 'NPMI Score']
    
    # Impostiamo i nomi delle features nel modello per il plot
    best_xgb.get_booster().feature_names = feature_names
    xgb.plot_importance(best_xgb, ax=ax1, importance_type='gain', title='XGBoost Feature Importance (Gain)', color='#4ade80')
    ax1.set_ylabel('Features LKG')
    
    # 2. Confusion Matrix
    # Calcoliamo le previsioni binarie sul Test Set
    y_pred = best_xgb.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax2)
    ax2.set_title('Confusion Matrix (Candidati Sbagliati vs Corretti)')
    ax2.set_xlabel('Predicted Label')
    ax2.set_ylabel('True Label')
    
    plt.tight_layout()
    plt.show()
else:
    print("Esegui prima la cella di Tuning di XGBoost per valorizzare 'best_xgb'.")